In [ ]:
!pip install bertopic sentence-transformers sentencepiece transformers

import pandas as pd
import numpy as np
import torch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, T5EncoderModel

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df= pd.read_csv("/content/aib_phishing.csv")

In [ ]:
print(df.shape)

(2002, 2)


In [ ]:
print (df.head())

                                              emails  is_phishing
0  More people started buying drugs on the darkne...            0
1  MICROSOFT OUTLOOK NOTIFICATION Your e-mail box...            1
2  Ben and Tom, Congratulations on the paper comi...            0
3  Dear PayPal Member, As part of our security me...            1
4  Dear vBankWorks user, Fraudulent activity has ...            1


In [ ]:
print(df["is_phishing"].value_counts())

is_phishing
1    1019
0     983
Name: count, dtype: int64


In [ ]:
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(df["emails"].astype(str).tolist())
topic_info = topic_model.get_topic_info()
print(topic_info)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    Topic  ...                                Representative_Docs
0      -1  ...  [Dear Washington Mutual customer, In accordanc...
1       0  ...  [Hi Peck et al. A little late but I'd like to ...
2       1  ...  [Dear User;  Your Mail Quota Has Exceeded The ...
3       2  ...  [why?=20 Jordan Kaplan National Finance Direct...
4       3  ...  [Your account will be suspended !     We are  ...
5       4  ...  [Dear Webmail Account User  This Email is from...
6       5  ...  [NSM to march in Omaha bringing attention to t...
7       6  ...  [Dear Valued Customer : As part of our securit...
8       7  ...  [From: Cornell Security Alert Notice Sent: Mon...
9       8  ...  [Los Angeles Times Entertainment Breaking News...
10      9  ...  [ eBay                 sent this message to yo...
11     10  ...  [05/10/2016 07:30 AM EDT By Natasha Korecki (u...
12     11  ...  [Hi everyone, Please see the agenda below for ...
13     12  ...  [[Image result for blackboard logo] Dear User ...
14     13 

In [ ]:
topic_info = topic_info[topic_info["Topic"] != -1]
display(topic_info)


,Topic,Count,Name,Representation,Representative_Docs
1,0,197,0_the_of_in_that,"[the, of, in, that, and, is, to, it, be, we]",[Hi Peck et al. A little late but I'd like to ...
2,1,76,1_mailbox_your_quota_exceeded,"[mailbox, your, quota, exceeded, storage, admi...",[Dear User; Your Mail Quota Has Exceeded The ...
3,2,75,2_national_finance_director_committee,"[national, finance, director, committee, 312, ...",[why?=20 Jordan Kaplan National Finance Direct...
4,3,64,3_paypal_account_your_access,"[paypal, account, your, access, we, to, you, l...",[Your account will be suspended ! We are ...
5,4,63,4_webmail_account_email_your,"[webmail, account, email, your, this, message,...",[Dear Webmail Account User This Email is from...
6,5,59,5_nsm_the_and_schoep,"[nsm, the, and, schoep, commander, of, nationa...",[NSM to march in Omaha bringing attention to t...
7,6,57,6_bank_america_your_account,"[bank, america, your, account, online, banking...",[Dear Valued Customer : As part of our securit...
8,7,55,7_cornell_university_account_your,"[cornell, university, account, your, netid, em...",[From: Cornell Security Alert Notice Sent: Mon...
9,8,53,8_angeles_los_awards_breaking,"[angeles, los, awards, breaking, times, 2014, ...",[Los Angeles Times Entertainment Breaking News...
10,9,48,9_ebay_item_trademarks_policy,"[ebay, item, trademarks, policy, member, inc, ...",[ eBay sent this message to yo...


In [ ]:
print("Number of topics:", len(topic_info))
print("Most Common Topic:")
display(topic_info.sort_values("Count", ascending=False).head(1))

Number of topics: 47
Most Common Topic:


,Topic,Count,Name,Representation,Representative_Docs
1,0,197,0_the_of_in_that,"[the, of, in, that, and, is, to, it, be, we]",[Hi Peck et al. A little late but I'd like to ...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["emails"].astype(str).tolist(),
    df["is_phishing"].astype(int),
    test_size=0.2,
    random_state=42,
    stratify=df["is_phishing"].astype(int)
)

In [ ]:
tdif = TfidfVectorizer(max_features=1000)
X_train_tdif = tdif.fit_transform(X_train).toarray()
X_test_tdif = tdif.transform(X_test).toarray()
ngram = TfidfVectorizer(ngram_range=(3,5), max_features=10000)
X_train_ngram = ngram.fit_transform(X_train).toarray()
X_test_ngram = ngram.transform(X_test).toarray()
print("TF-IDF and N-Gram completed")

TF-IDF and N-Gram completed


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5 = T5EncoderModel.from_pretrained("t5-small").to(device)
t5.eval()

def t5_embeddings(texts):
    embeddings = []

    for i in range(0, len(texts), 16):
        batch = texts[i:i + 16]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)


        with torch.no_grad():
            output = t5(**inputs)

        mask = inputs["attention_mask"].unsqueeze(-1)
        vectors = (output.last_hidden_state * mask).sum(1)/mask.sum(1)
        embeddings.extend(vectors.cpu().numpy())

    return np.array(embeddings)


X_train_t5 = t5_embeddings(X_train)


X_test_t5 = t5_embeddings(X_test)

print(" t5 Completed!")

Using device: cpu


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

 t5 Completed!


In [ ]:

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_sentence = sentence_model.encode(
    X_train,
    batch_size=32,
    show_progress_bar=True,
)
X_test_sentence = sentence_model.encode(
    X_test,
    batch_size=32,
    show_progress_bar=True,
)
print("Sentence Completed!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Sentence Completed!


In [ ]:


algorithims = {
    "Random Forest": RandomForestClassifier(),
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(),

}

methods = {
    "Tfidf": (X_train_tdif, X_test_tdif),
    "N-gram": (X_train_ngram, X_test_ngram),
    "T5": (X_train_t5, X_test_t5),
    "Sentence": (X_train_sentence, X_test_sentence)
}

results = pd.DataFrame(
    index=algorithims.keys(),
    columns=methods.keys(),
    dtype=float
)

for name, algorithim in algorithims.items():
    for method, (X_train_method, X_test_method) in methods.items():
      algorithim.fit(X_train_method, y_train)
      predictions = algorithim.predict(X_test_method)

      results.loc[name, method] = f1_score(y_test, y_pred=predictions)

results = results.astype(float)
results.index.name = "Algorithim"
results.columns.name = "Method"

display(results.round(3))


Method,Tfidf,N-gram,T5,Sentence
Algorithim,,,,
Random Forest,0.963,0.872,0.936,0.947
Logistic Regression,0.943,0.920,0.968,0.946
Naive Bayes,0.880,0.931,0.906,0.902
Decision Tree,0.932,0.888,0.866,0.848


In [ ]:
table = results.copy()
table["Average"] = table.mean(axis=1)
table.loc["Average"] = table.mean(axis=0)
table = table.reset_index().round(3).astype(str)

display(table)

chatbot = pipeline(
    "table-question-answering",
    model="google/tapas-base-finetuned-wtq"
)

def ask(question):
  answer = chatbot(
      table=table,
      query=question
  )
  print("Question:", question)
  print("Answer:", answer["answer"])

  ask("Which algorithim has the highest average score?")
  ask("Which method has the highest average score?")



Method,Algorithim,Tfidf,N-gram,T5,Sentence,Average
0,Random Forest,0.963,0.872,0.936,0.947,0.93
1,Logistic Regression,0.943,0.92,0.968,0.946,0.944
2,Naive Bayes,0.88,0.931,0.906,0.902,0.905
3,Decision Tree,0.932,0.888,0.866,0.848,0.883
4,Average,0.929,0.903,0.919,0.911,0.915


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

In [ ]:
question = input("Ask a question: ")
answer = chatbot(
    table=table,
    query=question
)
print("Question:", question)
print("Answer:", answer["answer"])


Ask a question: Which algorithm has the best score across all four?
Question: Which algorithm has the best score across all four?
Answer: Random Forest


In [ ]:
print("Best Algorithim:")
print(results.mean(axis=1).idxmax())
print("Best Embedding Method:")
print(results.mean(axis=0).idxmax())

Best Algorithim:
Logistic Regression
Best Embedding Method:
Tfidf


In [ ]:
question = input("Ask a question: ")
answer = chatbot(
    table=table,
    query=question
)
print("Question:", question)
print("Answer:", answer["answer"])

Ask a question: Which text vectorizing/embedding method has the highest score across all four?
Question: Which text vectorizing/embedding method has the highest score across all four?
Answer: Random Forest


In [ ]:
answer1 = chatbot(
    table=table,
    query="Which algorithm has the highest average score?"
)
print("Question: Which algorithm has the highest average score?")
print("Answer:", answer1["answer"])

answer2 = chatbot(
    table=table,
    query="Which embedding method has the highest average score?"
)
print("Question: Which embedding method has the highest average score?")
print("Answer:", answer2["answer"])

Question: Which algorithm has the highest average score?
Answer: Logistic Regression
Question: Which embedding method has the highest average score?
Answer: Logistic Regression
